In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

%matplotlib inline
sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

### This code loads the Adult dataset and assigns column names.

In [ ]:
columns = [
    "age","workclass","fnlwgt","education","education-num",
    "marital-status","occupation","relationship","race","sex",
    "capital-gain","capital-loss","hours-per-week",
    "native-country","income"
]

df = pd.read_csv("adult.data", names=columns, sep=",", skipinitialspace=True)
df.head()

### This code checks the dataset shape, column types, and non-null values.

In [ ]:
print("Shape:", df.shape)
df.info()

### This code shows descriptive statistics for numerical columns.

In [ ]:
df.describe()

### This code replaces '?' values with NaN so missing values can be handled properly.

In [ ]:
df.replace("?", np.nan, inplace=True)
df.isnull().sum()

### This code visualizes the target variable distribution.

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x="income", data=df)
plt.title("Income Distribution")
plt.show()

### This code creates predictors X and target y.

In [ ]:
X = df.drop("income", axis=1)
y = df["income"]

X.head()

### This code identifies categorical and numerical columns for preprocessing.

In [ ]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numeric_cols)

### This code splits the data into training and testing sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

### This code builds a preprocessing pipeline: fill missing values, encode categorical features, and scale numerical features.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols),
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_cols)
    ]
)

preprocessor

### This code creates and trains the KNN model using k = 1.

In [ ]:
knn = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier(n_neighbors=1))
])

knn.fit(X_train, y_train)

### This code generates predictions using the trained KNN model.

In [ ]:
predictions = knn.predict(X_test)
predictions[:10]

### This code evaluates the KNN model using accuracy, confusion matrix, and classification report.

In [ ]:
print("Accuracy:", accuracy_score(y_test, predictions))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))
print("\nClassification Report:")
print(classification_report(y_test, predictions))

### This code visualizes the confusion matrix for the KNN model.

In [ ]:
cm = confusion_matrix(y_test, predictions, labels=knn.named_steps["classifier"].classes_)

plt.figure(figsize=(6,4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=knn.named_steps["classifier"].classes_,
    yticklabels=knn.named_steps["classifier"].classes_
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

### This code tests different K values and stores the error rate for each K.

In [ ]:
error_rate = []

for k in range(1, 31):
    knn_k = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", KNeighborsClassifier(n_neighbors=k))
    ])

    knn_k.fit(X_train, y_train)
    pred_k = knn_k.predict(X_test)
    error_rate.append(np.mean(pred_k != y_test))

error_rate[:10]

### This code plots the error rate for different K values.

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(range(1,31), error_rate, marker="o", linestyle="dashed")
plt.title("Error Rate vs K Value")
plt.xlabel("K")
plt.ylabel("Error Rate")
plt.xticks(range(1,31))
plt.show()

### This code selects the best K value based on the lowest error rate.

In [ ]:
best_k = range(1, 31)[int(np.argmin(error_rate))]
print("Best K:", best_k)
print("Lowest Error Rate:", min(error_rate))

### This code trains the final KNN model using the best K value.

In [ ]:
final_knn = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier(n_neighbors=best_k))
])

final_knn.fit(X_train, y_train)

final_predictions = final_knn.predict(X_test)
final_predictions[:10]

### This code evaluates the final KNN model after choosing the best K value.

In [ ]:
print("Final Accuracy:", accuracy_score(y_test, final_predictions))
print("\nFinal Confusion Matrix:")
print(confusion_matrix(y_test, final_predictions))
print("\nFinal Classification Report:")
print(classification_report(y_test, final_predictions))

### This code visualizes the final confusion matrix.

In [ ]:
final_cm = confusion_matrix(y_test, final_predictions, labels=final_knn.named_steps["classifier"].classes_)

plt.figure(figsize=(6,4))
sns.heatmap(
    final_cm, annot=True, fmt="d", cmap="Greens",
    xticklabels=final_knn.named_steps["classifier"].classes_,
    yticklabels=final_knn.named_steps["classifier"].classes_
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Final KNN Confusion Matrix")
plt.show()